# Lab Exercise: SQL Analysis with Polars

In this lab, you'll practice SQL queries using Polars' built-in SQL functionality. Complete each exercise by writing the appropriate SQL query.

In [3]:
import polars as pl

# Load data
airlines = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airlines.csv')
airports = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airports.csv')
flights = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_flights.csv', null_values=['NA'])
planes = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_planes.csv', null_values=['NA'])
weather = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_weather.csv', null_values=['NA'], schema_overrides={'precip': pl.Float64, 'visib': pl.Float64})

flights = flights.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))
weather = weather.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))

# Create SQL context
ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

print("Setup complete! Tables available:")
print(ctx.execute("SHOW TABLES"))

Setup complete! Tables available:
shape: (5, 1)
┌──────────┐
│ name     │
│ ---      │
│ str      │
╞══════════╡
│ airlines │
│ airports │
│ flights  │
│ planes   │
│ weather  │
└──────────┘


/tmp/ipython-input-2504310122.py:14: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


## Exercise 1: Basic Queries

### 1.1 Find all unique carriers in the airlines table

In [4]:
# Write your SQL query here
result = ctx.execute("""
SELECT DISTINCT carrier
FROM airlines
ORDER BY carrier
""")

print(result)


shape: (16, 1)
┌─────────┐
│ carrier │
│ ---     │
│ str     │
╞═════════╡
│ 9E      │
│ AA      │
│ AS      │
│ B6      │
│ DL      │
│ …       │
│ UA      │
│ US      │
│ VX      │
│ WN      │
│ YV      │
└─────────┘


### 1.2 Find the top 10 destinations by number of flights

In [5]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  dest,
  COUNT(*) AS flight_count
FROM flights
GROUP BY dest
ORDER BY flight_count DESC
LIMIT 10
""")
print(result)

shape: (10, 2)
┌──────┬──────────────┐
│ dest ┆ flight_count │
│ ---  ┆ ---          │
│ str  ┆ u32          │
╞══════╪══════════════╡
│ ORD  ┆ 17283        │
│ ATL  ┆ 17215        │
│ LAX  ┆ 16174        │
│ BOS  ┆ 15508        │
│ MCO  ┆ 14082        │
│ CLT  ┆ 14064        │
│ SFO  ┆ 13331        │
│ FLL  ┆ 12055        │
│ MIA  ┆ 11728        │
│ DCA  ┆ 9705         │
└──────┴──────────────┘


### 1.3 Find all flights that departed more than 2 hours late (120 minutes)

In [6]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  year, month, day,
  carrier, flight, origin, dest,
  dep_delay
FROM flights
WHERE dep_delay > 120
ORDER BY dep_delay DESC
""")
print(result)

shape: (9_723, 8)
┌──────┬───────┬─────┬─────────┬────────┬────────┬──────┬───────────┐
│ year ┆ month ┆ day ┆ carrier ┆ flight ┆ origin ┆ dest ┆ dep_delay │
│ ---  ┆ ---   ┆ --- ┆ ---     ┆ ---    ┆ ---    ┆ ---  ┆ ---       │
│ i64  ┆ i64   ┆ i64 ┆ str     ┆ i64    ┆ str    ┆ str  ┆ i64       │
╞══════╪═══════╪═════╪═════════╪════════╪════════╪══════╪═══════════╡
│ 2013 ┆ 1     ┆ 9   ┆ HA      ┆ 51     ┆ JFK    ┆ HNL  ┆ 1301      │
│ 2013 ┆ 6     ┆ 15  ┆ MQ      ┆ 3535   ┆ JFK    ┆ CMH  ┆ 1137      │
│ 2013 ┆ 1     ┆ 10  ┆ MQ      ┆ 3695   ┆ EWR    ┆ ORD  ┆ 1126      │
│ 2013 ┆ 9     ┆ 20  ┆ AA      ┆ 177    ┆ JFK    ┆ SFO  ┆ 1014      │
│ 2013 ┆ 7     ┆ 22  ┆ MQ      ┆ 3075   ┆ JFK    ┆ CVG  ┆ 1005      │
│ …    ┆ …     ┆ …   ┆ …       ┆ …      ┆ …      ┆ …    ┆ …         │
│ 2013 ┆ 9     ┆ 2   ┆ B6      ┆ 1103   ┆ JFK    ┆ SJU  ┆ 121       │
│ 2013 ┆ 9     ┆ 11  ┆ EV      ┆ 5940   ┆ EWR    ┆ SAV  ┆ 121       │
│ 2013 ┆ 9     ┆ 12  ┆ AA      ┆ 2297   ┆ LGA    ┆ MIA  ┆ 121       │
│ 

## Exercise 2: Aggregation

### 2.1 Calculate the average departure delay for each origin airport

In [7]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  origin,
  AVG(dep_delay) AS avg_delay
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY origin
ORDER BY avg_delay DESC
""")
print(result)

shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘


### 2.2 Find the busiest month of the year

Count the number of flights per month and find which month has the most flights.

In [8]:
# First, let's check what columns are available
result = ctx.execute("""
    SELECT *
    FROM flights
    LIMIT 5
""")
print(result)

result = ctx.execute("""
SELECT
  month,
  COUNT(*) AS flights
FROM flights
GROUP BY month
ORDER BY flights DESC
LIMIT 1
""")
print(result)

shape: (5, 19)
┌──────┬───────┬─────┬──────────┬───┬──────────┬──────┬────────┬─────────────────────────┐
│ year ┆ month ┆ day ┆ dep_time ┆ … ┆ distance ┆ hour ┆ minute ┆ time_hour               │
│ ---  ┆ ---   ┆ --- ┆ ---      ┆   ┆ ---      ┆ ---  ┆ ---    ┆ ---                     │
│ i64  ┆ i64   ┆ i64 ┆ i64      ┆   ┆ i64      ┆ i64  ┆ i64    ┆ datetime[μs, UTC]       │
╞══════╪═══════╪═════╪══════════╪═══╪══════════╪══════╪════════╪═════════════════════════╡
│ 2013 ┆ 1     ┆ 1   ┆ 517      ┆ … ┆ 1400     ┆ 5    ┆ 15     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 533      ┆ … ┆ 1416     ┆ 5    ┆ 29     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 542      ┆ … ┆ 1089     ┆ 5    ┆ 40     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 544      ┆ … ┆ 1576     ┆ 5    ┆ 45     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 554      ┆ … ┆ 762      ┆ 6    ┆ 0      ┆ 2013-01-01 11:00:00 UTC │
└──────┴───────┴─────┴──────────┴───┴──────────┴──────┴────────┴───────────

### 2.3 Calculate the on-time performance rate for each carrier

Consider a flight on-time if the departure delay is <= 15 minutes.

In [9]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  carrier,
  AVG(CASE WHEN dep_delay <= 15 THEN 1 ELSE 0 END) AS on_time_rate
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY carrier
ORDER BY on_time_rate DESC
""")
print(result)

shape: (16, 2)
┌─────────┬──────────────┐
│ carrier ┆ on_time_rate │
│ ---     ┆ ---          │
│ str     ┆ f64          │
╞═════════╪══════════════╡
│ HA      ┆ 0.929825     │
│ US      ┆ 0.878227     │
│ AS      ┆ 0.867978     │
│ AA      ┆ 0.840713     │
│ DL      ┆ 0.836812     │
│ …       ┆ …            │
│ FL      ┆ 0.733291     │
│ WN      ┆ 0.731027     │
│ F9      ┆ 0.718475     │
│ YV      ┆ 0.713761     │
│ EV      ┆ 0.695381     │
└─────────┴──────────────┘


## Exercise 3: Joins

### 3.1 List all flights with their airline names (not just carrier codes)

Show the first 20 flights with carrier code, airline name, flight number, origin, and destination.

In [10]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  f.carrier,
  a.name AS airline_name,
  f.flight,
  f.origin,
  f.dest
FROM flights AS f
JOIN airlines AS a
  ON a.carrier = f.carrier
LIMIT 20
""")
print(result)

shape: (20, 5)
┌─────────┬────────────────────────┬────────┬────────┬──────┐
│ carrier ┆ airline_name           ┆ flight ┆ origin ┆ dest │
│ ---     ┆ ---                    ┆ ---    ┆ ---    ┆ ---  │
│ str     ┆ str                    ┆ i64    ┆ str    ┆ str  │
╞═════════╪════════════════════════╪════════╪════════╪══════╡
│ UA      ┆ United Air Lines Inc.  ┆ 1545   ┆ EWR    ┆ IAH  │
│ UA      ┆ United Air Lines Inc.  ┆ 1714   ┆ LGA    ┆ IAH  │
│ AA      ┆ American Airlines Inc. ┆ 1141   ┆ JFK    ┆ MIA  │
│ B6      ┆ JetBlue Airways        ┆ 725    ┆ JFK    ┆ BQN  │
│ DL      ┆ Delta Air Lines Inc.   ┆ 461    ┆ LGA    ┆ ATL  │
│ …       ┆ …                      ┆ …      ┆ …      ┆ …    │
│ B6      ┆ JetBlue Airways        ┆ 1806   ┆ JFK    ┆ BOS  │
│ UA      ┆ United Air Lines Inc.  ┆ 1187   ┆ EWR    ┆ LAS  │
│ B6      ┆ JetBlue Airways        ┆ 371    ┆ LGA    ┆ FLL  │
│ MQ      ┆ Envoy Air              ┆ 4650   ┆ LGA    ┆ ATL  │
│ B6      ┆ JetBlue Airways        ┆ 343    ┆ EWR    ┆ 

### 3.2 Find the average age of planes for each carrier

Hint: The planes table has a `year` column for manufacture year. Calculate age based on 2013.

In [11]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  f.carrier,
  AVG(2013 - p.year) AS avg_plane_age
FROM flights AS f
JOIN planes  AS p
  ON f.tailnum = p.tailnum
WHERE p.year IS NOT NULL
GROUP BY f.carrier
ORDER BY avg_plane_age DESC
""")
print(result)


shape: (16, 2)
┌─────────┬───────────────┐
│ carrier ┆ avg_plane_age │
│ ---     ┆ ---           │
│ str     ┆ f64           │
╞═════════╪═══════════════╡
│ MQ      ┆ 35.319        │
│ AA      ┆ 25.869426     │
│ DL      ┆ 16.372169     │
│ UA      ┆ 13.207691     │
│ FL      ┆ 11.385829     │
│ …       ┆ …             │
│ B6      ┆ 6.686702      │
│ F9      ┆ 4.87874       │
│ VX      ┆ 4.473643      │
│ AS      ┆ 3.33662       │
│ HA      ┆ 1.548387      │
└─────────┴───────────────┘


### 3.3 Find flights that experienced both departure delays and bad weather

Join flights with weather data and find flights where departure delay > 30 minutes and either wind_speed > 20 or precip > 0.1

In [12]:
# First, explore the weather table structure
result = ctx.execute("""
    SELECT *
    FROM weather
    LIMIT 5
""")
print(result)
result = ctx.execute("""
SELECT
  f.year, f.month, f.day, f.hour,
  f.origin, f.dest,
  f.dep_delay,
  w.wind_speed, w.precip
FROM flights AS f
JOIN weather AS w
  ON f.origin = w.origin
 AND f.time_hour = w.time_hour
WHERE f.dep_delay > 30
  AND (w.wind_speed > 20 OR w.precip > 0.1)
ORDER BY f.dep_delay DESC
""")
print(result)

shape: (5, 15)
┌────────┬──────┬───────┬─────┬───┬────────┬──────────┬───────┬─────────────────────────┐
│ origin ┆ year ┆ month ┆ day ┆ … ┆ precip ┆ pressure ┆ visib ┆ time_hour               │
│ ---    ┆ ---  ┆ ---   ┆ --- ┆   ┆ ---    ┆ ---      ┆ ---   ┆ ---                     │
│ str    ┆ i64  ┆ i64   ┆ i64 ┆   ┆ f64    ┆ f64      ┆ f64   ┆ datetime[μs, UTC]       │
╞════════╪══════╪═══════╪═════╪═══╪════════╪══════════╪═══════╪═════════════════════════╡
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.0   ┆ 10.0  ┆ 2013-01-01 06:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.3   ┆ 10.0  ┆ 2013-01-01 07:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.5   ┆ 10.0  ┆ 2013-01-01 08:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.2   ┆ 10.0  ┆ 2013-01-01 09:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1011.9   ┆ 10.0  ┆ 2013-01-01 10:00:00 UTC │
└────────┴──────┴───────┴─────┴───┴────────┴──────────┴───────┴──────────────────────

## Exercise 4: Advanced Queries

### 4.1 Find the most popular aircraft types (by number of flights)

Join flights with planes to get manufacturer and model information. Show top 10.

In [13]:
# Write your SQL query here
result = ctx.execute("""
SELECT
  p.manufacturer,
  p.model,
  COUNT(*) AS flights
FROM flights AS f
JOIN planes  AS p
  ON f.tailnum = p.tailnum
GROUP BY p.manufacturer, p.model
ORDER BY flights DESC
LIMIT 10
""")
print(result)

shape: (10, 3)
┌───────────────────────────────┬─────────────────┬─────────┐
│ manufacturer                  ┆ model           ┆ flights │
│ ---                           ┆ ---             ┆ ---     │
│ str                           ┆ str             ┆ u32     │
╞═══════════════════════════════╪═════════════════╪═════════╡
│ AIRBUS                        ┆ A320-232        ┆ 31278   │
│ EMBRAER                       ┆ EMB-145LR       ┆ 28027   │
│ EMBRAER                       ┆ ERJ 190-100 IGW ┆ 23716   │
│ AIRBUS INDUSTRIE              ┆ A320-232        ┆ 14553   │
│ EMBRAER                       ┆ EMB-145XR       ┆ 14051   │
│ BOEING                        ┆ 737-824         ┆ 13809   │
│ BOMBARDIER INC                ┆ CL-600-2D24     ┆ 11807   │
│ BOEING                        ┆ 737-7H4         ┆ 10389   │
│ BOEING                        ┆ 757-222         ┆ 9150    │
│ MCDONNELL DOUGLAS AIRCRAFT CO ┆ MD-88           ┆ 8932    │
└───────────────────────────────┴─────────────────┴────

### 4.2 Analyze route performance

Find the top 10 routes (origin-destination pairs) with:
- Total number of flights
- Average departure delay
- Percentage of flights delayed more than 30 minutes

Include airport names, not just codes.

In [14]:
# Write your SQL query here
result = ctx.execute("""
WITH route_stats AS (
  SELECT
    f.origin,
    f.dest,
    COUNT(*)                            AS flight_count,
    AVG(f.dep_delay)                    AS avg_dep_delay,
    100.0 * AVG(CASE WHEN f.dep_delay > 30 THEN 1 ELSE 0 END)
                                        AS pct_dep_delay_gt30
  FROM flights AS f
  GROUP BY f.origin, f.dest
)
SELECT
  rs.origin,
  ao.name AS origin_name,
  rs.dest,
  ad.name AS dest_name,
  rs.flight_count,
  rs.avg_dep_delay,
  rs.pct_dep_delay_gt30
FROM route_stats AS rs
LEFT JOIN airports AS ao ON rs.origin = ao.faa
LEFT JOIN airports AS ad ON rs.dest   = ad.faa
ORDER BY rs.flight_count DESC
LIMIT 10
""")
print(result)

shape: (10, 7)
┌────────┬─────────────────┬──────┬────────────────┬──────────────┬───────────────┬────────────────┐
│ origin ┆ origin_name     ┆ dest ┆ dest_name      ┆ flight_count ┆ avg_dep_delay ┆ pct_dep_delay_ │
│ ---    ┆ ---             ┆ ---  ┆ ---            ┆ ---          ┆ ---           ┆ gt30           │
│ str    ┆ str             ┆ str  ┆ str            ┆ u32          ┆ f64           ┆ ---            │
│        ┆                 ┆      ┆                ┆              ┆               ┆ f64            │
╞════════╪═════════════════╪══════╪════════════════╪══════════════╪═══════════════╪════════════════╡
│ JFK    ┆ John F Kennedy  ┆ LAX  ┆ Los Angeles    ┆ 11262        ┆ 8.522508      ┆ 9.829515       │
│        ┆ Intl            ┆      ┆ Intl           ┆              ┆               ┆                │
│ LGA    ┆ La Guardia      ┆ ATL  ┆ Hartsfield     ┆ 10263        ┆ 11.448621     ┆ 12.247881      │
│        ┆                 ┆      ┆ Jackson        ┆              ┆         

In [15]:
# Write your SQL query here
result = ctx.execute("""
WITH route_stats AS (
  SELECT
    f.origin,
    f.dest,
    COUNT(*)                            AS flight_count,
    AVG(f.dep_delay)                    AS avg_dep_delay,
    100.0 * AVG(CASE WHEN f.dep_delay > 30 THEN 1 ELSE 0 END)
                                        AS pct_dep_dep_gt30
  FROM flights AS f
  GROUP BY f.origin, f.dest
)
SELECT
  rs.origin,
  ao.name AS origin_name,
  rs.dest,
  ad.name AS dest_name,
  rs.flight_count,
  rs.avg_dep_delay,
  rs.pct_dep_dep_gt30
FROM route_stats AS rs
LEFT JOIN airports AS ao ON rs.origin = ao.faa
LEFT JOIN airports AS ad ON rs.dest   = ad.faa
ORDER BY rs.avg_dep_delay DESC
LIMIT 10
""")
print(result)

shape: (10, 7)
┌────────┬─────────────────┬──────┬────────────────┬──────────────┬───────────────┬────────────────┐
│ origin ┆ origin_name     ┆ dest ┆ dest_name      ┆ flight_count ┆ avg_dep_delay ┆ pct_dep_dep_gt │
│ ---    ┆ ---             ┆ ---  ┆ ---            ┆ ---          ┆ ---           ┆ 30             │
│ str    ┆ str             ┆ str  ┆ str            ┆ u32          ┆ f64           ┆ ---            │
│        ┆                 ┆      ┆                ┆              ┆               ┆ f64            │
╞════════╪═════════════════╪══════╪════════════════╪══════════════╪═══════════════╪════════════════╡
│ EWR    ┆ Newark Liberty  ┆ LGA  ┆ La Guardia     ┆ 1            ┆ null          ┆ 0.0            │
│        ┆ Intl            ┆      ┆                ┆              ┆               ┆                │
│ EWR    ┆ Newark Liberty  ┆ TYS  ┆ Mc Ghee Tyson  ┆ 323          ┆ 41.818471     ┆ 37.4613        │
│        ┆ Intl            ┆      ┆                ┆              ┆         

## Bonus: Compare with Polars

### Choose one of the queries above and implement it using Polars

This will help you understand the relationship between SQL and Polars operations.

In [17]:
# Implementing Exercise 4.2 in Polars:
origin_airports = airports.select(
    pl.col("faa").alias("origin"),
    pl.col("name").alias("origin_name")
)
dest_airports = airports.select(
    pl.col("faa").alias("dest"),
    pl.col("name").alias("dest_name")
)

polars_route_perf = (
    flights
    .group_by(["origin", "dest"])
    .agg([
        pl.count().alias("flight_count"),
        pl.col("dep_delay").mean().alias("avg_dep_delay"),
        (pl.when(pl.col("dep_delay") > 30).then(1).otherwise(0).mean() * 100.0)
            .alias("pct_dep_delay_gt30")
    ])
    .join(origin_airports, on="origin", how="left")
    .join(dest_airports,   on="dest",   how="left")
    .sort("flight_count", descending=True)
    .head(10)
)

print(polars_route_perf)


shape: (10, 7)
┌────────┬──────┬──────────────┬───────────────┬─────────────────┬────────────────┬────────────────┐
│ origin ┆ dest ┆ flight_count ┆ avg_dep_delay ┆ pct_dep_delay_g ┆ origin_name    ┆ dest_name      │
│ ---    ┆ ---  ┆ ---          ┆ ---           ┆ t30             ┆ ---            ┆ ---            │
│ str    ┆ str  ┆ u32          ┆ f64           ┆ ---             ┆ str            ┆ str            │
│        ┆      ┆              ┆               ┆ f64             ┆                ┆                │
╞════════╪══════╪══════════════╪═══════════════╪═════════════════╪════════════════╪════════════════╡
│ JFK    ┆ LAX  ┆ 11262        ┆ 8.522508      ┆ 9.829515        ┆ John F Kennedy ┆ Los Angeles    │
│        ┆      ┆              ┆               ┆                 ┆ Intl           ┆ Intl           │
│ LGA    ┆ ATL  ┆ 10263        ┆ 11.448621     ┆ 12.247881       ┆ La Guardia     ┆ Hartsfield     │
│        ┆      ┆              ┆               ┆                 ┆          

/tmp/ipython-input-4074484213.py:15: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("flight_count"),
